<a href="https://colab.research.google.com/github/shin-noda/deep-learning-with-python/blob/main/Chapter18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Installing KerasTuner
!pip install keras-tuner -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 4.0 MB/s eta 0:00:00


In [8]:
import keras
from keras import layers

def build_model(hp):
    # Sample hyperparameter values from the hp object. After sampling,
    # these values (such as the "units" variable here) are just regular Python constants.
    units = hp.Int(name="units", min_value=16, max_value=64, step=16)
    model = keras.Sequential(
        [
            layers.Dense(units, activation="relu"),
            layers.Dense(10, activation="softmax"),
        ]
    )

    # Different kinds of hyperparameters are available: Int, Float, Boolean, Choice.
    optimizer = hp.Choice(name="optimizer", values=["rmsprop", "adam"])
    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    # The function returns a compiled model.
    return model

In [9]:
import keras_tuner as kt

class SimpleMLP(kt.HyperModel):
    # Thanks to the object-oriented approach, we can configure model
    # constants as constructor arguments (instead of hardcoding them in
    # the model-building function).
    def __init__(self, num_classes):
        self.num_classes = num_classes


    # The build method is identical to our prior build_model standalone function.
    def build(self, hp):
        units = hp.Int(name="units", min_value=16, max_value=64, step=16)
        model = keras.Sequential(
            [
                layers.Dense(units, activation="relu"),
                layers.Dense(self.num_classes, activation="softmax"),
            ]
        )

        optimizer = hp.Choice(name="optimizer", values=["rmsprop", "adam"])
        model.compile(
            optimizer=optimizer,
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )

        return model

hypermodel = SimpleMLP(num_classes=10)

In [10]:
tuner = kt.BayesianOptimization(
    # Specifies the model-building function (or hypermodel instance)
    build_model,

    # Specifies the metric that the tuner will seek to optimize. Always
    # specify validation metrics, since the goal of the serch process
    # is to find models that generalize.
    objective="val_accuracy",

    # Maximum number of different model configurations ("trials") to
    # try before ending the search
    max_trials=20,

    # To reduce metrics variance, you can train the same model multiple times and
    # average the results. executions_per_trial is how many training rounds (executions)
    # to run for each model configuration (trial).
    executions_per_trial=2,

    # Where to store search logs
    directory="mnist_kt_test",

    # Whether to overwrite data in the directory to start a new search.
    # Set this to True if you've modified the model-building function or to False
    # to resume a previously started search with the same model-buidling function.
    overwrite=True,
)

In [11]:
tuner.search_space_summary()

Search space summary
Default search space size: 2
units (Int)
{'default': None, 'conditions': [], 'min_value': 16, 'max_value': 64, 'step': 16, 'sampling': 'linear'}
optimizer (Choice)
{'default': 'rmsprop', 'conditions': [], 'values': ['rmsprop', 'adam'], 'ordered': False}


In [12]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train = x_train.reshape((-1, 28 * 28)).astype("float32") / 255
x_test = x_test.reshape((-1, 28 * 28)).astype("float32") / 255

# Reserves these for later
x_train_full = x_train[:]
y_train_full = y_train[:]

# Sets aside a validation set
num_val_samples = 10000
x_train, x_val = x_train[:-num_val_samples], x_train[-num_val_samples:]
y_train, y_val = y_train[:-num_val_samples], y_train[-num_val_samples:]
callbacks = [
    # Uses a large number of epochs (you don't know in advance how many epochs each model
    # will need) and uses an EarlyStopping callback to stop training when you start overfitting
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5),
]

# This takes the same arguments as fit() (it simply passes them down to
# fit() for each new model).
tuner.search(
    x_train,
    y_train,
    batch_size=128,
    epochs=100,
    validation_data=(x_val, y_val),
    callbacks=callbacks,
    verbose=2,
)

Trial 20 Complete [00h 01m 06s]
val_accuracy: 0.9750499725341797

Best val_accuracy So Far: 0.9762000143527985
Total elapsed time: 00h 23m 17s


In [13]:
top_n = 4

# Returns a list of HyperParameters objects, which you can pass to the model-building function
best_hps = tuner.get_best_hyperparameters(top_n)

In [14]:
def get_best_epochs(hp):
    model = build_model(hp)
    callbacks = [
        keras.callbacks.EarlyStopping(
            # Note the very high patience value.
            monitor="val_loss", mode="min", patience=10
        )
    ]

    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_val, y_val),
        epochs=100,
        batch_size=128,
        callbacks=callbacks,
    )

    val_loss_per_epoch = history.history["val_loss"]
    best_epoch = val_loss_per_epoch.index(min(val_loss_per_epoch)) + 1
    print(f"Best epoch: {best_epoch}")

    return best_epoch

In [16]:
def get_best_trained_model(hp):
    best_epoch = get_best_epochs(hp)
    model = build_model(hp)
    model.fit(
        x_train_full, y_train_full, batch_size=128, epochs=int(best_epoch * 1.2)
    )

    return model

best_models = []

for hp in best_hps:
    model = get_best_trained_model(hp)
    model.evaluate(x_test, y_test)
    best_models.append(model)

Epoch 1/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8856 - loss: 0.4251 - val_accuracy: 0.9283 - val_loss: 0.2455
Epoch 2/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9335 - loss: 0.2282 - val_accuracy: 0.9391 - val_loss: 0.2092
Epoch 3/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9490 - loss: 0.1765 - val_accuracy: 0.9531 - val_loss: 0.1652
Epoch 4/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9585 - loss: 0.1436 - val_accuracy: 0.9619 - val_loss: 0.1356
Epoch 5/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9654 - loss: 0.1208 - val_accuracy: 0.9671 - val_loss: 0.1172
Epoch 6/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9698 - loss: 0.1044 - val_accuracy: 0.9689 - val_loss: 0.1117
Epoch 7/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9731 - loss: 0.0920 - val_accuracy: 0.9695 - val_loss: 0.1093
Epoch 8/100
391/391 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.9763 - loss: 0.0818 - val_accu

In [17]:
# best_models = tuner.get_best_models(top_n)

In [18]:
# Model ensembling

In [20]:
# # Uses four different models to compute initial predictions
# preds_a = model_a.predict(x_val)
# preds_b = model_b.predict(x_val)
# preds_c = model_c.predict(x_val)
# preds_d = model_d.predict(x_val)

# # This new predictions array should be more accurate than any of the initial ones.
# final_preds = 0.25 * (preds_a + preds_b + preds_c + preds_d)

In [21]:
# preds_a = model_a.predict(x_val)
# preds_b = model_b.predict(x_val)
# preds_c = model_c.predict(x_val)
# preds_d = model_d.predict(x_val)

# # These weights (0.5, 0.25, 0.1, 0.15) are assumed to be learned empirically.
# final_preds = 0.5 * preds_a + 0.25 * preds_b + 0.1 * preds_c + 0.15 * preds_d

In [23]:
# device_mesh = keras.distribution.DeviceMesh(
#     # We assume eight devices, organized as 2 x 4 grid.
#     shape=(2, 4),

#     # It's convenient to give your axes meaningful names
#     axis_names=["data", "model"],
# )